# Notebook 20 — Project Runtime Prototype

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/20_project_runtime_prototype.ipynb)

The Castalia Runtime capstone starts with a notebook-native prototype for **Castalia Mentor**. The goal is to make the serving stack explicit: define the request contract, compare a local path with an optimized path, and leave behind artifacts that can grow into a real service.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What you will build**
- Understand **Capstone framing**
- Understand **Step 1: Prepare artifact folders and helpers**
- Understand **Step 2: Define the Castalia Mentor request contract**
- Understand **Step 3: Describe the runtime inventory**


## What you will build

- a request and response contract for Castalia Mentor
- a local runtime path and an optimized serving path
- a simple notebook-native service skeleton with deterministic routing
- a runtime comparison table for the prototype workloads
- an artifact layout that later notebooks can reuse for routing and deployment

## Capstone framing

Project: **Castalia Runtime**

Goal: serve Castalia Mentor through a notebook-native runtime stack that includes local and optimized serving paths, runtime comparison, routing, structured outputs, observability, and deployment handoff.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

## Step 1: Prepare artifact folders and helpers

We start by giving the runtime prototype a predictable file layout. This mirrors the style used across the rest of the systems track: simple JSON and CSV outputs, transparent directories, and minimal hidden machinery.

In [ ]:
random.seed(20)

ARTIFACT_DIR = ARTIFACT_ROOT / "project_20_runtime_prototype"
for folder_name in ["contracts", "manifests", "logs", "samples"]:
    (ARTIFACT_DIR / folder_name).mkdir(parents=True, exist_ok=True)

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Define the Castalia Mentor request contract

The serving layer should know what kind of request it is handling before any model call begins. The contract captures the messages, target latency, and whether the caller needs structured JSON instead of free-form text.

In [ ]:
from pydantic import BaseModel, Field

class MentorMessage(BaseModel):
    role: str
    content: str

class RuntimeRequest(BaseModel):
    request_id: str
    mode: str = Field(default="chat")
    target_latency_ms: int = 1400
    require_json: bool = False
    messages: list[MentorMessage]
    metadata: dict[str, Any] = Field(default_factory=dict)

class RuntimeResponse(BaseModel):
    request_id: str
    selected_path: str
    output_text: str
    structured_payload: dict[str, Any] | None = None
    metrics: dict[str, float]

request_contract_schema = RuntimeRequest.model_json_schema()
print(json.dumps(request_contract_schema, indent=2)[:1600])

## Step 3: Describe the runtime inventory

The prototype exposes two primary paths: a local notebook path and an optimized GPU server path. We also note an advanced routing runtime as future context so the inventory already matches the broader course vocabulary.

In [ ]:
runtime_inventory = [
    {
        "path": "local_notebook",
        "backend": "llama.cpp",
        "deployment_shape": "GGUF model on notebook or laptop hardware",
        "p50_latency_ms": 1480,
        "max_context": 8192,
        "structured_support": "post-parse JSON repair",
        "best_for": "single-user iteration and low-cost fallback",
    },
    {
        "path": "optimized_server",
        "backend": "vLLM",
        "deployment_shape": "OpenAI-compatible GPU serving endpoint",
        "p50_latency_ms": 780,
        "max_context": 32768,
        "structured_support": "guided JSON or parser-backed responses",
        "best_for": "shared traffic and latency-sensitive routes",
    },
    {
        "path": "advanced_router",
        "backend": "SGLang",
        "deployment_shape": "advanced scheduling and structured serving",
        "p50_latency_ms": 690,
        "max_context": 65536,
        "structured_support": "schema-aware serving patterns",
        "best_for": "long-context or heavy structured output workloads",
    },
]

runtime_df = pd.DataFrame(runtime_inventory)
runtime_df

## Step 4: Normalize a small set of Castalia Mentor requests

Capstones work better when the request mix is concrete. These prototype tasks represent common Castalia Mentor workloads: explanation, extraction, planning, and grounded policy guidance.

In [ ]:
mentor_tasks = [
    {
        "task": "mentor_chat_short",
        "mode": "chat",
        "target_latency_ms": 1400,
        "require_json": False,
        "user_prompt": "Explain why continuous batching helps shared LLM serving.",
    },
    {
        "task": "mentor_extract_json",
        "mode": "extract",
        "target_latency_ms": 1100,
        "require_json": True,
        "user_prompt": "Return a JSON object with runtime, bottleneck, and recommended fix for a slow local inference setup.",
    },
    {
        "task": "mentor_plan_rollout",
        "mode": "planning",
        "target_latency_ms": 1500,
        "require_json": True,
        "user_prompt": "Draft a rollout plan for moving Castalia Mentor from notebook testing to a shared GPU endpoint.",
    },
    {
        "task": "mentor_policy_long",
        "mode": "grounded_chat",
        "target_latency_ms": 1800,
        "require_json": False,
        "user_prompt": "Summarize the deployment handoff checklist and mention the observability items the on-call engineer should verify first.",
    },
]

def build_request(task, index):
    return RuntimeRequest(
        request_id=f"castalia-{index:03d}",
        mode=task["mode"],
        target_latency_ms=task["target_latency_ms"],
        require_json=task["require_json"],
        messages=[
            MentorMessage(role="system", content="You are Castalia Mentor, an open-source systems coach."),
            MentorMessage(role="user", content=task["user_prompt"]),
        ],
        metadata={"task": task["task"]},
    )

sample_requests = [build_request(task, index) for index, task in enumerate(mentor_tasks, start=1)]
request_preview_df = pd.DataFrame([
    {
        "request_id": request.request_id,
        "task": request.metadata["task"],
        "target_latency_ms": request.target_latency_ms,
        "require_json": request.require_json,
    }
    for request in sample_requests
])
request_preview_df

## Step 5: Estimate local versus optimized runtime fit

Before we write a service skeleton, we can reason about path selection with lightweight estimates. The exact numbers will change in real deployments, but the notebook still teaches the right mental model: prompt size, output size, memory footprint, and SLA fit.

In [ ]:
def estimate_token_count(text):
    return max(1, math.ceil(len(text.split()) * 1.35))

def estimate_plan(path_name, request):
    prompt_tokens = sum(estimate_token_count(message.content) for message in request.messages)
    output_tokens = 160 if request.require_json else 220
    if path_name == "local_notebook":
        prefill_ms = 210 + prompt_tokens * 1.15
        decode_ms = 320 + output_tokens * 1.45
        memory_gb = 5.6
    elif path_name == "optimized_server":
        prefill_ms = 120 + prompt_tokens * 0.42
        decode_ms = 180 + output_tokens * 0.88
        memory_gb = 15.0
    else:
        prefill_ms = 110 + prompt_tokens * 0.36
        decode_ms = 175 + output_tokens * 0.74
        memory_gb = 17.5
    estimated_total_ms = round(prefill_ms + decode_ms, 2)
    return {
        "request_id": request.request_id,
        "task": request.metadata["task"],
        "path": path_name,
        "prompt_tokens": prompt_tokens,
        "output_tokens": output_tokens,
        "estimated_total_ms": estimated_total_ms,
        "estimated_memory_gb": memory_gb,
        "fits_sla": estimated_total_ms <= request.target_latency_ms,
    }

plan_rows = []
for request in sample_requests:
    for path_name in ["local_notebook", "optimized_server"]:
        plan_rows.append(estimate_plan(path_name, request))

plan_df = pd.DataFrame(plan_rows)
plan_df

## Step 6: Build a notebook-native service skeleton

The class below is intentionally small. It chooses a path, synthesizes an answer, records simple metrics, and stays close to the explicit request contract defined above.

In [ ]:
class CastaliaRuntimePrototype:
    def __init__(self, inventory):
        self.inventory = {item["path"]: item for item in inventory}
        self.request_log = []

    def select_path(self, request):
        candidates = [estimate_plan("local_notebook", request), estimate_plan("optimized_server", request)]
        long_prompt = any(len(message.content) > 180 for message in request.messages)
        if request.require_json or long_prompt:
            selected = next(item for item in candidates if item["path"] == "optimized_server")
            reason = "structured_or_long_prompt"
        else:
            sla_candidates = [item for item in candidates if item["fits_sla"]]
            pool = sla_candidates if sla_candidates else candidates
            selected = min(pool, key=lambda item: item["estimated_total_ms"])
            reason = "lowest_latency_within_budget" if sla_candidates else "best_available_path"
        return {"choice": selected, "reason": reason}

    def render_output(self, request, path_name):
        user_text = next((message.content for message in reversed(request.messages) if message.role == "user"), "")
        label = "local path" if path_name == "local_notebook" else "optimized path"
        answer = "Castalia Mentor runtime prototype via " + label + ": " + user_text[:180]
        structured_payload = None
        if request.require_json:
            structured_payload = {
                "answer": answer,
                "route": path_name,
                "contract_version": "2026-06",
                "request_id": request.request_id,
            }
        return answer, structured_payload

    def serve(self, request):
        selection = self.select_path(request)
        answer, structured_payload = self.render_output(request, selection["choice"]["path"])
        metrics = {
            "estimated_total_ms": float(selection["choice"]["estimated_total_ms"]),
            "estimated_memory_gb": float(selection["choice"]["estimated_memory_gb"]),
            "prompt_tokens": float(selection["choice"]["prompt_tokens"]),
            "output_tokens": float(selection["choice"]["output_tokens"]),
        }
        response = RuntimeResponse(
            request_id=request.request_id,
            selected_path=selection["choice"]["path"],
            output_text=answer,
            structured_payload=structured_payload,
            metrics=metrics,
        )
        self.request_log.append({
            "request_id": request.request_id,
            "task": request.metadata["task"],
            "selected_path": response.selected_path,
            "reason": selection["reason"],
            **metrics,
        })
        return response

prototype = CastaliaRuntimePrototype(runtime_inventory)
prototype

## Step 7: Serve prototype requests

We can now run the service skeleton against the prototype task set. The responses are deterministic so the notebook stays stable, but the routing logic is the same kind of logic a real service would use.

In [ ]:
responses = [prototype.serve(request) for request in sample_requests]

response_df = pd.DataFrame([
    {
        "request_id": response.request_id,
        "task": request.metadata["task"],
        "selected_path": response.selected_path,
        "estimated_total_ms": response.metrics["estimated_total_ms"],
        "structured": response.structured_payload is not None,
    }
    for request, response in zip(sample_requests, responses)
])
response_df

## Step 8: Compare the local path with the optimized path

The prototype should leave behind a visible comparison. This makes the later benchmark notebook feel like a refinement of the same idea instead of a disconnected exercise.

In [ ]:
comparison_rows = []
for task_name in sorted(response_df["task"].unique()):
    local_row = plan_df[(plan_df["task"] == task_name) & (plan_df["path"] == "local_notebook")].iloc[0]
    optimized_row = plan_df[(plan_df["task"] == task_name) & (plan_df["path"] == "optimized_server")].iloc[0]
    chosen_row = response_df[response_df["task"] == task_name].iloc[0]
    comparison_rows.append({
        "task": task_name,
        "local_ms": local_row["estimated_total_ms"],
        "optimized_ms": optimized_row["estimated_total_ms"],
        "latency_savings_ms": round(local_row["estimated_total_ms"] - optimized_row["estimated_total_ms"], 2),
        "selected_path": chosen_row["selected_path"],
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("latency_savings_ms", ascending=False)
comparison_df

## Step 9: Define the prototype artifact layout

Artifact discipline matters early. By writing down what belongs in contracts, manifests, logs, and samples, we make the capstone easier to extend into routing, observability, and deployment handoff.

In [ ]:
artifact_layout = {
    "contracts": ["request_contract.json", "response_examples.json"],
    "manifests": ["runtime_inventory.json", "artifact_layout.json", "service_skeleton.json"],
    "logs": ["request_log.jsonl"],
    "samples": ["runtime_comparison.csv", "prototype_summary.csv"],
}

artifact_layout_df = pd.DataFrame([
    {"folder": folder_name, "files": ", ".join(file_names)}
    for folder_name, file_names in artifact_layout.items()
])
artifact_layout_df

## Step 10: Export prototype contracts and manifests

The notebook should leave behind reusable evidence, not only transient cell output. These files become inputs to the next capstone stages.

In [ ]:
write_json(ARTIFACT_DIR / "contracts" / "request_contract.json", request_contract_schema)
write_json(ARTIFACT_DIR / "contracts" / "response_examples.json", [response.model_dump() for response in responses])
write_json(ARTIFACT_DIR / "manifests" / "runtime_inventory.json", runtime_inventory)
write_json(ARTIFACT_DIR / "manifests" / "artifact_layout.json", artifact_layout)
write_json(
    ARTIFACT_DIR / "manifests" / "service_skeleton.json",
    {
        "class_name": "CastaliaRuntimePrototype",
        "supported_paths": ["local_notebook", "optimized_server"],
        "selection_rules": [
            "structured_or_long_prompt -> optimized_server",
            "otherwise choose the lowest-latency path that fits the SLA",
        ],
    },
)
(ARTIFACT_DIR / "samples" / "runtime_comparison.csv").write_text(comparison_df.to_csv(index=False), encoding="utf-8")
(ARTIFACT_DIR / "samples" / "prototype_summary.csv").write_text(response_df.to_csv(index=False), encoding="utf-8")
(ARTIFACT_DIR / "logs" / "request_log.jsonl").write_text("\n".join(json.dumps(row) for row in prototype.request_log), encoding="utf-8")
print("Wrote prototype artifacts:", sorted(str(path.relative_to(ARTIFACT_DIR)) for path in ARTIFACT_DIR.rglob("*") if path.is_file()))

## Step 11: Sketch backend adapters for real open-source runtimes

The notebook prototype is useful precisely because it can later point at real backends without changing the request contract. We keep the launch commands explicit and open-source only.

In [ ]:
backend_commands = {
    "local_notebook": [
        "llama-server",
        "-m",
        str(CACHE_DIR / "gguf" / "Castalia-Mentor-Q4.gguf"),
        "--host", "127.0.0.1",
        "--port", "8008",
        "-c", "8192",
    ],
    "optimized_server": [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-7B-Instruct",
        "--host", "127.0.0.1",
        "--port", "8010",
        "--max-model-len", "32768",
    ],
    "advanced_router": [
        "python", "-m", "sglang.launch_server",
        "--model-path", "Qwen/Qwen2.5-7B-Instruct",
        "--host", "127.0.0.1",
        "--port", "8011",
    ],
}

adapter_df = pd.DataFrame([
    {"path": path_name, "launch_command": " ".join(command)}
    for path_name, command in backend_commands.items()
])
adapter_df

## Recap

You now have a notebook-native Castalia Runtime prototype with a request contract, a local path, an optimized path, a service skeleton, and reusable artifacts. Notebook 21 turns those prototype assumptions into benchmark-driven routing and structured-output policy.

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** benchmark with a different model size and compare throughput. Document what changes and why.

**Exercise 2 — Build:** add a monitoring metric to the serving pipeline. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** simulate a failure scenario and verify the system recovers.

## 📝 Key Takeaways

- **What you will build** — revisit this section for reference
- **Capstone framing** — revisit this section for reference
- **Step 1: Prepare artifact folders and helpers** — revisit this section for reference
- **Step 2: Define the Castalia Mentor request contract** — revisit this section for reference
- **Step 3: Describe the runtime inventory** — revisit this section for reference


## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the systems/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [19 Eval Driven Rollouts Regressions And Incident Playbooks](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/19_eval_driven_rollouts_regressions_and_incident_playbooks.ipynb) | ➡️ **Next:** [21 Project Benchmark Routing And Structured Outputs](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/21_project_benchmark_routing_and_structured_outputs.ipynb)